# 02 · 用 NumPy 從零手刻神經網路

**對應教學 App**：🧠 神經網路是什麼

**不用 PyTorch、不用 TensorFlow**，只用 NumPy 把整個神經網路實作出來：
前向傳播 → 損失函數 → 反向傳播 → 梯度下降。

**為什麼要做這個**：面試官問「`loss.backward()` 到底在算什麼」時，
手刻過的人和只會呼叫 API 的人，答案完全不同層次。

---

## 0. 準備

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Microsoft YaHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

np.random.seed(42)
print("✅ 準備完成")

## 1. 造一份「一條直線分不開」的資料

兩個同心圓。用一條直線絕對分不開 —— 這正是我們需要多層網路的理由。

In [ ]:
def make_circles(n=600, noise=0.12, seed=42):
    rng = np.random.default_rng(seed)
    half = n // 2
    theta = rng.uniform(0, 2 * np.pi, n)
    r = np.r_[rng.uniform(0.0, 1.1, half), rng.uniform(1.9, 3.0, half)]
    X = np.c_[r * np.cos(theta), r * np.sin(theta)]
    X += rng.normal(0, noise, X.shape)
    y = np.r_[np.zeros(half), np.ones(half)].reshape(-1, 1)
    idx = rng.permutation(n)
    return X[idx], y[idx]


X, y = make_circles()

plt.figure(figsize=(6, 6))
plt.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], s=18, c="#2563eb", label="類別 0（內圈）")
plt.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], s=18, c="#dc2626", label="類別 1（外圈）")
plt.legend(); plt.title("目標：把內圈和外圈分開"); plt.axis("equal"); plt.grid(alpha=.3)
plt.show()

print("X 形狀:", X.shape, "  y 形狀:", y.shape)

## 2. 零件一：激活函數

**ReLU**：`max(0, x)`，隱藏層用。
**Sigmoid**：壓成 0～1 的機率，輸出層用。

每個函數都要寫「**它自己**」和「**它的導數**」——
導數是反向傳播時要用的。

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    """ReLU 的導數：z > 0 時是 1，否則是 0"""
    return (z > 0).astype(float)

def sigmoid(z):
    # 減掉最大值防止 exp 溢位（數值穩定的標準寫法）
    return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))


# 畫出來看看
zs = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(zs, relu(zs), lw=3, color="#2563eb", label="ReLU(z)")
axes[0].plot(zs, relu_grad(zs), lw=2, ls="--", color="#dc2626", label="ReLU 的導數")
axes[0].set_title("ReLU"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(zs, sigmoid(zs), lw=3, color="#2563eb", label="Sigmoid(z)")
axes[1].plot(zs, sigmoid(zs) * (1 - sigmoid(zs)), lw=2, ls="--", color="#dc2626",
             label="Sigmoid 的導數")
axes[1].set_title("Sigmoid（注意導數最大只有 0.25 → 這就是梯度消失的原因）")
axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3. 零件二：損失函數

二分類用 **Binary Cross-Entropy（二元交叉熵）**：

$$L = -\frac{1}{m}\sum \left[ y\log(\hat{y}) + (1-y)\log(1-\hat{y}) \right]$$

**白話**：答案是 1 時，模型猜越接近 1 損失越小；答案是 0 時反過來。
猜錯得越離譜，`log` 會讓懲罰爆炸性成長。

In [ ]:
def bce_loss(y_true, y_pred, eps=1e-9):
    y_pred = np.clip(y_pred, eps, 1 - eps)     # 防止 log(0) = -inf
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


# 感受一下：答案是 1 的時候，猜不同機率的損失
for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"正確答案 = 1，模型猜 {p:.2f} → 損失 {bce_loss(np.array([[1.]]), np.array([[p]])):.4f}")

## 4. 組裝：兩層神經網路

架構：`輸入(2) → 隱藏層(16) + ReLU → 輸出(1) + Sigmoid`

### 前向傳播

```
z1 = X @ W1 + b1      形狀 (m, 16)
a1 = ReLU(z1)
z2 = a1 @ W2 + b2     形狀 (m, 1)
a2 = Sigmoid(z2)      ← 最終預測機率
```

### 反向傳播（連鎖律，從後往前）

```
dz2 = (a2 - y) / m           ← BCE + Sigmoid 合併後的漂亮結果
dW2 = a1.T @ dz2
db2 = sum(dz2)
da1 = dz2 @ W2.T
dz1 = da1 * relu_grad(z1)    ← 通過 ReLU 往回傳
dW1 = X.T @ dz1
db1 = sum(dz1)
```

💡 `dz2 = a2 - y` 這麼簡潔不是巧合 —— 這是 Sigmoid 搭配交叉熵時，
數學上自然化簡出來的結果。**這也是為什麼分類不用 MSE 而用交叉熵**。

In [ ]:
class TinyNeuralNetwork:
    def __init__(self, n_in=2, n_hidden=16, n_out=1, seed=42):
        rng = np.random.default_rng(seed)
        # He 初始化：專為 ReLU 設計，讓每層輸出的變異數保持穩定
        self.W1 = rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden))
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.normal(0, np.sqrt(2 / n_hidden), (n_hidden, n_out))
        self.b2 = np.zeros((1, n_out))

    # ---------- 前向傳播 ----------
    def forward(self, X):
        self.X  = X
        self.z1 = X @ self.W1 + self.b1
        self.a1 = relu(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    # ---------- 反向傳播 ----------
    def backward(self, y):
        m = len(y)
        dz2 = (self.a2 - y) / m
        self.dW2 = self.a1.T @ dz2
        self.db2 = dz2.sum(axis=0, keepdims=True)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * relu_grad(self.z1)
        self.dW1 = self.X.T @ dz1
        self.db1 = dz1.sum(axis=0, keepdims=True)

    # ---------- 梯度下降更新 ----------
    def step(self, lr):
        self.W1 -= lr * self.dW1
        self.b1 -= lr * self.db1
        self.W2 -= lr * self.dW2
        self.b2 -= lr * self.db2


print("✅ 網路類別定義完成")

## 5. 檢查梯度算對了沒（Gradient Check）

**這是手刻神經網路最重要的一步。**

原理：導數的定義是 `(f(w+ε) - f(w-ε)) / 2ε`。
用這個「數值方法」算一次梯度，跟我們反向傳播算的比較。
**兩者應該幾乎一樣**（誤差 < 1e-7）。

In [ ]:
net = TinyNeuralNetwork(n_hidden=8)

# 反向傳播算的梯度
net.forward(X[:50])
net.backward(y[:50])
grad_backprop = net.dW1[0, 0]

# 數值方法算的梯度
eps = 1e-5
net.W1[0, 0] += eps
loss_plus = bce_loss(y[:50], net.forward(X[:50]))
net.W1[0, 0] -= 2 * eps
loss_minus = bce_loss(y[:50], net.forward(X[:50]))
net.W1[0, 0] += eps                       # 還原

grad_numeric = (loss_plus - loss_minus) / (2 * eps)

print(f"反向傳播算的梯度: {grad_backprop:.10f}")
print(f"數值方法算的梯度: {grad_numeric:.10f}")
print(f"相對誤差        : {abs(grad_backprop - grad_numeric) / (abs(grad_backprop) + 1e-12):.2e}")
print()
print("✅ 誤差小於 1e-6 就代表反向傳播寫對了。")

## 6. 訓練

In [ ]:
def train(n_hidden=16, lr=0.5, epochs=3000, seed=42, verbose=True):
    net = TinyNeuralNetwork(n_hidden=n_hidden, seed=seed)
    history = {"loss": [], "acc": []}

    for ep in range(epochs):
        pred = net.forward(X)                  # ① 前向
        loss = bce_loss(y, pred)               # ② 算損失
        net.backward(y)                        # ③ 反向
        net.step(lr)                           # ④ 更新

        history["loss"].append(loss)
        history["acc"].append(float(((pred > 0.5) == y).mean()))

        if verbose and ep % 500 == 0:
            print(f"epoch {ep:5d}　loss {loss:.4f}　acc {history['acc'][-1]:.3f}")

    return net, history


net, hist = train()
print()
print(f"最終正確率：{hist['acc'][-1]:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist["loss"], color="#dc2626", lw=2)
axes[0].set_title("損失 Loss"); axes[0].set_xlabel("epoch"); axes[0].grid(alpha=.3)
axes[1].plot(hist["acc"], color="#16a34a", lw=2)
axes[1].set_title("正確率"); axes[1].set_xlabel("epoch"); axes[1].set_ylim(0, 1); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. 看它學到什麼樣的分界線

In [ ]:
def plot_boundary(net, title=""):
    gx, gy = np.meshgrid(np.linspace(-3.6, 3.6, 250), np.linspace(-3.6, 3.6, 250))
    zz = net.forward(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

    plt.figure(figsize=(7, 6.5))
    plt.contourf(gx, gy, zz, levels=25, cmap="RdBu_r", alpha=.7)
    plt.colorbar(label="預測機率")
    plt.contour(gx, gy, zz, levels=[0.5], colors="black", linewidths=2.5)
    plt.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], s=14, c="#1e40af", edgecolors="w", lw=.4)
    plt.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], s=14, c="#991b1b", edgecolors="w", lw=.4)
    plt.title(title); plt.axis("equal")
    plt.show()


plot_boundary(net, f"16 顆隱藏神經元學到的分界線（黑線 = 0.5 的等高線）")

## 8. 實驗：神經元數量的影響

**這格會跑幾秒鐘。** 看看神經元太少 / 剛好 / 太多，分界線長什麼樣。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4.6))

for ax, h in zip(axes, [1, 2, 8, 64]):
    n, hh = train(n_hidden=h, epochs=2500, verbose=False)
    gx, gy = np.meshgrid(np.linspace(-3.6, 3.6, 160), np.linspace(-3.6, 3.6, 160))
    zz = n.forward(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

    ax.contourf(gx, gy, zz, levels=20, cmap="RdBu_r", alpha=.7)
    ax.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], s=6, c="#1e40af")
    ax.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], s=6, c="#991b1b")
    ax.set_title(f"{h} 顆神經元　正確率 {hh['acc'][-1]:.1%}")
    ax.axis("equal"); ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout(); plt.show()

print("👉 1～2 顆：只能畫直線 → 欠擬合")
print("👉 8 顆以上：畫得出圓形 → 剛好")
print("👉 64 顆：邊界開始出現多餘的彎曲 → 已有過擬合的傾向")

## 9. 實驗：學習率的影響

In [ ]:
plt.figure(figsize=(9, 5))

for lr, color in [(0.01, "#94a3b8"), (0.1, "#2563eb"), (0.5, "#16a34a"),
                  (3.0, "#d97706"), (10.0, "#dc2626")]:
    _, h = train(lr=lr, epochs=1500, verbose=False)
    plt.plot(h["loss"], label=f"lr = {lr}", color=color, lw=2)

plt.xlabel("epoch"); plt.ylabel("損失"); plt.yscale("log")
plt.title("學習率的影響：太小走不動，太大會震盪甚至發散")
plt.legend(); plt.grid(alpha=.3)
plt.show()

## 10. 對照：用 PyTorch 寫同一件事

同樣的網路，PyTorch 只要 12 行 —— 因為 `loss.backward()` 幫你做掉了
我們在第 4 格手寫的所有反向傳播。

**但你現在知道它在算什麼了。**

In [ ]:
import torch
import torch.nn as nn

Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(3000):
    optimizer.zero_grad()        # ① 清梯度（忘了寫會出大錯）
    out = model(Xt)              # ② 前向
    loss = criterion(out, yt)
    loss.backward()              # ③ 反向 ← 我們手刻的那一大段，它一行做完
    optimizer.step()             # ④ 更新

with torch.no_grad():
    acc = ((torch.sigmoid(model(Xt)) > 0.5).float() == yt).float().mean()

print(f"PyTorch 版正確率：{acc:.1%}")
print(f"我們手刻版正確率：{hist['acc'][-1]:.1%}")
print()
print("👉 結果一樣。差別只在誰幫你算微分。")

---

## 🎯 動手改改看

1. **加一層**：把網路改成 `2 → 16 → 16 → 1`。
   你需要新增 `W3`、`b3`，並在 backward 多推導一層。**這是最好的練習。**

2. **把 ReLU 換成 Sigmoid**（隱藏層也用 sigmoid），重跑第 6 格。
   → loss 下降會變慢很多。這就是**梯度消失**的實際體感。

3. **把 He 初始化改成 `rng.normal(0, 0.01, ...)`**（很小的初始值）。
   → 訓練會變得很難收斂。初始化真的很重要。

4. **加入 L2 正則化**：在 `step()` 裡改成
   `self.W1 -= lr * (self.dW1 + lambda_ * self.W1)`，觀察分界線變化。

5. **改成 mini-batch 訓練**：每次只用 32 筆資料算梯度，而不是全部 600 筆。
   → loss 曲線會變得比較毛躁，但通常泛化更好。

## 📝 下一步

`03_CNN影像分類.ipynb`